In [1]:
import os
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split

In [2]:
os.chdir("../")

In [3]:
from src.functions import remove_stopwords_punctuation, remove_outliers

In [4]:
df = pd.read_csv("./data/buscape.csv")

In [5]:
df

,original_index,review_text,review_text_processed,review_text_tokenized,polarity,rating,kfold_polarity,kfold_rating
0,4_55516,"Estou muito satisfeito, o visor é melhor do qu...","estou muito satisfeito, o visor e melhor do qu...","['estou', 'muito', 'satisfeito', 'visor', 'mel...",1.0,4,1,1
1,minus_1_105339,"""muito boa\n\nO que gostei: preco\n\nO que não...","""muito boa\n\no que gostei: preco\n\no que nao...","['muito', 'boa', 'que', 'gostei', 'preco', 'qu...",1.0,5,1,1
2,23_382139,"Rápida, ótima qualidade de impressão e fácil d...","rapida, otima qualidade de impressao e facil d...","['rapida', 'otima', 'qualidade', 'de', 'impres...",1.0,5,1,1
3,2_446456,Produto de ótima qualidade em todos os quesito!,produto de otima qualidade em todos os quesito!,"['produto', 'de', 'otima', 'qualidade', 'em', ...",1.0,5,1,1
4,0_11324,Precisava comprar uma tv compatível com meu dv...,precisava comprar uma tv compativel com meu dv...,"['precisava', 'comprar', 'uma', 'tv', 'compati...",1.0,5,1,1
...,...,...,...,...,...,...,...,...
84986,1_422965,"Produto muito bom, simples e barato","produto muito bom, simples e barato","['produto', 'muito', 'bom', 'simples', 'barato']",1.0,5,10,10
84987,minus_1_150466,O esquema antigo de desmontagem e limpeza das ...,o esquema antigo de desmontagem e limpeza das ...,"['esquema', 'antigo', 'de', 'desmontagem', 'li...",NaN,3,-1,10
84988,0_414799,Esse jogo é muito maneiro é um jogo onde vc te...,esse jogo e muito maneiro e um jogo onde vc te...,"['esse', 'jogo', 'muito', 'maneiro', 'um', 'jo...",1.0,5,10,10
84989,0_389898,Muito bom e intuitivo!\n\nO que gostei: Educa ...,muito bom e intuitivo!\n\no que gostei: educa ...,"['muito', 'bom', 'intuitivo', 'que', 'gostei',...",NaN,3,-1,10


### 1. Train-Test Split

In [6]:
X_data = df.drop(columns=['polarity'])
y_data = df['polarity']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X_data, y_data, test_size=0.3, random_state=42)

## Conjunto de Treino

### 1. Limpeza de Dados
##### 1.1. Tratamento de Nulos

In [8]:
X_train.shape

(59493, 7)

In [9]:
X_train['review_text'].isnull().sum()

np.int64(1)

In [10]:
X_train = X_train.dropna(subset=['review_text'])

In [11]:
X_train.shape

(59492, 7)

In [12]:
# paridade de índices
y_train = y_train.loc[X_train.index]

In [13]:
y_train.shape

(59492,)

##### 1.2. Tratamento de Outliers

In [14]:
nltk.download("stopwords")
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\roger\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\roger\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\roger\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [15]:
X_train_token = pd.DataFrame()
X_train_token['token'] = X_train['review_text']
X_train_token

,token
69382,Ótimo aparelho. Para quem já usa ou irá experi...
11241,"Apesar de o acesso ser USB 2.0, tem boa veloci..."
80248,comparado ao produto que usava anteriormente é...
61220,Muito bom\n\nO que gostei: Muito instrutivo\n\...
76452,Produto excelente. Sua qualidade de imagem cha...
...,...
6265,acho lindo quero esse modelo de qualquer geito...
54886,todomundo vai querer um ipad como esse
76820,"foi um investimento muito bom, sem arrependime..."
860,"Considero um bom aparelho, com um conceito eco..."


In [16]:
X_train_token['token'] = X_train_token['token'].apply(word_tokenize)

In [17]:
X_train_token['token'] = X_train_token['token'].apply(lambda text: remove_stopwords_punctuation(text))

In [18]:
X_train_token['len_token'] = X_train_token['token'].apply(lambda text: len(text))

In [19]:
X_train_token

,token,len_token
69382,"[Ótimo, aparelho, Para, usa, irá, experimentar...",25
11241,"[Apesar, acesso, USB, 2.0, boa, velocidade, tr...",105
80248,"[comparado, produto, usava, anteriormente, sup...",40
61220,"[Muito, bom, O, gostei, Muito, instrutivo, O, ...",10
76452,"[Produto, excelente, Sua, qualidade, imagem, c...",18
...,...,...
6265,"[acho, lindo, quero, modelo, qualquer, geito, ...",19
54886,"[todomundo, vai, querer, ipad]",4
76820,"[investimento, bom, arrependimento, O, gostei,...",25
860,"[Considero, bom, aparelho, conceito, ecológico...",25


In [20]:
X_train_token = remove_outliers(X_train_token, 'len_token')

In [21]:
X_train_token

,token,len_token
69382,"[Ótimo, aparelho, Para, usa, irá, experimentar...",25
80248,"[comparado, produto, usava, anteriormente, sup...",40
61220,"[Muito, bom, O, gostei, Muito, instrutivo, O, ...",10
76452,"[Produto, excelente, Sua, qualidade, imagem, c...",18
51615,"[MDesign, moderno, todas, tecnologias, necessa...",21
...,...,...
6265,"[acho, lindo, quero, modelo, qualquer, geito, ...",19
54886,"[todomundo, vai, querer, ipad]",4
76820,"[investimento, bom, arrependimento, O, gostei,...",25
860,"[Considero, bom, aparelho, conceito, ecológico...",25


In [ ]:
# paridade de índices em X e y
X_train = X_train.loc[X_train_token.index]

In [24]:
y_train = y_train.loc[X_train_token.index]

In [25]:
print(X_train.shape, y_train.shape)

(55151, 7) (55151,)
